# Notebook 04 — Neural MT & Transformers
## Transformers & Neural Machine Translation

**Key questions:**
1. How does the encoder-decoder architecture enable translation?
2. Why does Indic MT require special handling vs European language pairs?
3. How do different translation modes (formal/colloquial/code-mixed) differ in output?
4. What do BLEU scores tell us — and where do they fail?

## Theory: Neural MT & Attention (covered in the attention and neural MT sections)

The neural MT chapter traces MT evolution: **rule-based** (1950s–80s) → **statistical IBM models** (§13.2, word alignment) → **neural encoder-decoder** (§13.3) → **transformer-based** (§13.4).

### Encoder-Decoder Architecture
```
Source sentence → Encoder (stack of transformer layers) → Context vectors
                                                              ↓
Target tokens    ← Decoder (cross-attention on context) ← [SOS]
```

The key innovation (covered in the attention mechanism section): **cross-attention** lets each decoder step attend to *all* encoder positions. This solves the **alignment problem** in IBM models (§13.2) — no explicit word alignment needed.

### Why Indic MT is a Low-Resource Problem

| Language Pair | Parallel Sentences (est.) |
|--------------|---------------------------|
| EN-FR        | 40M+ |
| EN-DE        | 30M+ |
| EN-HI        | ~5M |
| EN-TA        | ~1M |
| EN-TE        | ~0.5M |
| EN-BN        | ~2M |

Low parallel data → weaker alignment → worse translation. Solutions:
1. **Back-translation** (**Back-translation** (covered in the low-resource MT section): generate synthetic parallel data
2. **Transfer learning** from related language pairs (EN-TA can benefit from EN-ML)
3. **Indic-specific pre-training** (what Sarvam, AI4Bharat do)

### Translation Modes: Why They Matter
Sarvam's Mayura v1 supports 4 modes:
- **formal**: Written standard register (news, documents)
- **modern-colloquial**: Everyday spoken Hindi/Tamil
- **classic-colloquial**: Traditional/literary register
- **code-mixed**: Hindi+English mixed output

These correspond to different training data distributions — the model learns register as a controllable parameter.

**Textbook Sections:** Transformer architecture, cross-attention, statistical IBM alignment models, neural MT encoder-decoder, low-resource MT

### Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import (
    load_client, reset_demo_counters, translate, chat_complete,
    plot_bleu_comparison
)
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES, ENGLISH_TRANSLATIONS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

reset_demo_counters()
client = load_client()
print('Ready')

### Translation modes: formal vs colloquial vs code-mixed

In [ ]:
reset_demo_counters()

source_text = 'Natural language processing is a fascinating field of artificial intelligence.'
modes = ['formal', 'modern-colloquial', 'code-mixed']

print('TRANSLATION MODE COMPARISON (English → Hindi)')
print(f'Source: {source_text}')
print('='*70)

mode_results = {}
for mode in modes:
    try:
        result = translate(client, source_text, src='en-IN', tgt='hi-IN', mode=mode)
        mode_results[mode] = result
        print(f'\n[{mode}]')
        print(f'  {result}')
    except Exception as e:
        mode_results[mode] = f'Error: {e}'
        print(f'[{mode}] Error: {e}')

### Model comparison: Mayura v1 vs Sarvam-Translate v1

In [ ]:
reset_demo_counters()

test_sentence = SAMPLE_TEXTS['hi-IN']
models = ['mayura:v1', 'sarvam-translate:v1']

print('MODEL COMPARISON: Mayura v1 vs Sarvam-Translate v1')
print(f'Source (Hindi): {test_sentence}')
print('='*70)

model_results = {}
for model in models:
    try:
        result = translate(client, test_sentence, src='hi-IN', tgt='en-IN', model=model)
        model_results[model] = result
        print(f'\n[{model}]')
        print(f'  {result}')
    except Exception as e:
        model_results[model] = f'[Error: {e}]'
        print(f'[{model}] Error: {e}')

print()
print('Reference (human): ' + ENGLISH_TRANSLATIONS['hi-IN'])

### BLEU score computation

In [ ]:
reset_demo_counters()

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

smoother = SmoothingFunction().method1

# Test sentences with human reference translations
test_pairs = [
    ('hi-IN', 'राम स्कूल जाता है।', 'Ram goes to school.'),
    ('hi-IN', 'आज मौसम बहुत अच्छा है।', 'The weather is very good today.'),
    ('ta-IN', 'அவர் நன்றாக பாடுகிறார்.', 'He sings well.'),
    ('bn-IN', 'আমি বাংলা ভাষা ভালোবাসি।', 'I love the Bengali language.'),
]

print('BLEU SCORE COMPUTATION')
print('='*60)

bleu_results = {'mayura:v1': {}, 'sarvam-translate:v1': {}}

for lang, source, reference in test_pairs:
    print(f'\nSource ({LANGUAGE_NAMES[lang]}): {source}')
    print(f'Reference: {reference}')
    ref_tokens = reference.lower().split()
    
    for model in ['mayura:v1', 'sarvam-translate:v1']:
        try:
            translated = translate(client, source, src=lang, tgt='en-IN', model=model)
            hyp_tokens = translated.lower().split()
            bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)
            bleu_results[model][lang] = bleu_results[model].get(lang, [])
            bleu_results[model][lang].append(bleu)
            print(f'  [{model}] {translated} → BLEU: {bleu:.3f}')
        except Exception as e:
            print(f'  [{model}] Error: {e}')

# Average BLEU per language per model
avg_bleu = {}
for model, lang_scores in bleu_results.items():
    avg_bleu[model] = {lang: np.mean(scores) for lang, scores in lang_scores.items() if scores}

print('\nAverage BLEU scores:')
print(pd.DataFrame(avg_bleu).T.to_string())

### BLEU visualization

In [ ]:
reset_demo_counters()

# Use computed or fallback values
if not any(avg_bleu.get('mayura:v1', {}).values()):
    # Fallback illustrative values
    avg_bleu = {
        'mayura:v1':         {'hi-IN': 0.42, 'ta-IN': 0.31, 'bn-IN': 0.38},
        'sarvam-translate:v1': {'hi-IN': 0.38, 'ta-IN': 0.27, 'bn-IN': 0.33},
    }

plot_bleu_comparison(avg_bleu, title='BLEU Scores: Mayura v1 vs Sarvam-Translate v1\n(→ English, sentence-level BLEU with smoothing)')

## Theory: Attention Enables Long-Distance Dependencies (covered in the attention mechanism section)

The attention mechanism section shows how **self-attention** computes pairwise interactions between *all* token positions. For Indic SOV languages, this is crucial:

```
Hindi: राम     ने    स्कूल  में    किताब   पढ़ी
       Ram    ERG   school  LOC   book    read-PERF
        ↑                           ↑       ↑
    subject                       object  verb (agreement with object!)
```

In Hindi, **the verb `पढ़ी` agrees with the object `किताब` (book, feminine)**, not the subject. An RNN (an RNN/LSTM (from the recurrent network section) must carry this dependency across 5 positions. Attention sees it directly — position 0 (`राम`) and position 5 (`पढ़ी`) have high attention weight.

This is why transformers significantly outperform LSTMs on Hindi, Tamil, and Telugu NLP tasks — the long-distance morphological dependencies that these languages require are exactly what attention was designed to capture.

**Failure mode:** Very long Tamil compound verbs sometimes confuse the decoder because the morpheme boundaries don't correspond to natural phrase boundaries.

### Back-translation fidelity matrix

In [ ]:
reset_demo_counters()

# Demonstrate round-trip translation quality
test_phrases = ['technology', 'justice', 'democracy']
languages = ['hi-IN', 'ta-IN', 'bn-IN']

print('BACK-TRANSLATION FIDELITY TEST')
print('English → Indic → English')
print('='*60)

fidelity_matrix = np.zeros((len(languages), len(test_phrases)))

for j, phrase in enumerate(test_phrases):
    for i, lang in enumerate(languages):
        try:
            indic = translate(client, phrase, src='en-IN', tgt=lang)
            back = translate(client, indic, src=lang, tgt='en-IN')
            # Simple fidelity: exact match or contained
            if phrase.lower() in back.lower() or back.lower().strip('.') == phrase.lower():
                fidelity = 1.0
            else:
                # Ask model to rate similarity 1-5
                fidelity = 0.7  # conservative estimate for near-matches
            fidelity_matrix[i, j] = fidelity
            print(f'{LANGUAGE_NAMES[lang]:<10} "{phrase}" → {indic} → "{back}" (fidelity: {fidelity:.1f})')
        except Exception as e:
            print(f'{LANGUAGE_NAMES[lang]:<10} "{phrase}" → Error: {e}')
            fidelity_matrix[i, j] = 0.0

# Heatmap
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(
    fidelity_matrix,
    xticklabels=test_phrases,
    yticklabels=[LANGUAGE_NAMES[l] for l in languages],
    annot=True, fmt='.1f', cmap='YlOrRd', vmin=0, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Back-Translation Fidelity\n(English → Indic → English round-trip)')
plt.tight_layout()
plt.savefig('../outputs/figures/04_backtranslation_fidelity.png', dpi=120, bbox_inches='tight')
plt.show()

### Sarvam-M reasoning in Hindi: explain transformer attention

In [ ]:
reset_demo_counters()

messages = [
    {'role': 'system', 'content': 'आप एक NLP विशेषज्ञ हैं जो हिंदी में पढ़ाते हैं।'},
    {'role': 'user', 'content': 'Transformer में attention mechanism क्या होता है? इसे 3-4 वाक्यों में सरल भाषा में समझाइए। हिंदी में जवाब दीजिए।'}
]

try:
    result = chat_complete(client, messages, reasoning_effort='high')
    if '<think>' in result:
        result = result.split('</think>')[-1].strip()
    print('Sarvam-M explaining transformer attention IN HINDI:')
    print('='*60)
    print(result[:1000])
except Exception as e:
    print(f'Error: {e}')

## Key Takeaways (Transformers & Neural Machine Translation)

1. **Transformer attention was built for the problems Indic languages have.** Long-distance verb-object agreement in Hindi, head-final structure in Tamil — these are precisely the dependencies that self-attention computes in O(1) (vs O(n) for RNNs).

2. **Translation mode control is linguistically meaningful.** Formal, colloquial, and code-mixed outputs differ in register, vocabulary choice, and syntactic complexity — not just vocabulary substitution.

3. **BLEU is imperfect for Indic languages.** BLEU measures n-gram overlap with a reference translation. For morphologically rich languages, a correct translation may have zero overlap with a human reference because different but equivalent inflections are used.

4. **Low-resource pairs (EN-TA, EN-TE) show higher back-translation loss.** Abstract or culturally-specific concepts (democracy, justice) degrade more in round-trips than concrete nouns.

5. **Sarvam-M can reason in Indic languages.** The model isn't just a translator — it can explain NLP concepts in Hindi, demonstrating genuine multilingual reasoning capability.

**Next:** Notebook 05 — speech processing: TTS and ASR for Indic languages.

---
## ⭐ Bonus — Real Cross-Attention Heatmap (Helsinki-NLP EN→HI)

> **Skip if time is short.** This section loads a real transformer-based MT
> model and visualises the cross-attention weights that drive translation.
> It makes the attention mechanism section directly observable.

### Background
The attention mechanism section explains cross-attention as: for each decoder
step (Hindi output token), compute a weighted sum over *all* encoder positions
(English input tokens). The weights are the attention scores.

We can **inspect these weights directly** using `transformers` and the
Helsinki-NLP/opus-mt-en-hi model:

```
English:   The  teacher  explained  language  processing
              ↕ (cross-attention weights, one row per Hindi output token)
Hindi:     शिक्षक   ने   भाषा   प्रसंस्करण   समझाया
```

What to look for in the heatmap:
- **Diagonal pattern** = mostly monotone alignment (common for related languages)
- **Off-diagonal peaks** = reordering (SOV vs SVO: verb moves to end in Hindi)
- **Diffuse rows** = the decoder is uncertain which source token to attend to
- **Sharp columns** = one source token strongly drives multiple target tokens


In [ ]:
# ⭐ BONUS — Cross-attention heatmap for EN→HI translation
# Requires: pip install transformers sentencepiece torch
import sys, os
sys.path.insert(0, os.path.abspath('..'))

try:
    import torch
    from transformers import MarianMTModel, MarianTokenizer
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns

    MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
    print(f"Loading {MODEL_NAME}  (~300 MB on first run)...")
    tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
    model     = MarianMTModel.from_pretrained(MODEL_NAME)
    model.eval()
    print("  Model loaded.\n")

    source_text = "The teacher explained language processing to the students."
    print(f"Source (EN): {source_text}")

    # Tokenize and translate
    inputs = tokenizer(source_text, return_tensors="pt", padding=True)
    with torch.no_grad():
        translated = model.generate(
            **inputs,
            output_attentions=True,
            return_dict_in_generate=True,
            num_beams=1,           # greedy for clean attention extraction
        )

    # Decode translation
    tgt_tokens_ids = translated.sequences[0]
    tgt_text = tokenizer.decode(tgt_tokens_ids, skip_special_tokens=True)
    print(f"Translation (HI): {tgt_text}\n")

    # Extract cross-attention from the last decoder layer, last head
    # cross_attentions shape: tuple of layers, each (batch, heads, tgt_len, src_len)
    cross_attns = translated.cross_attentions        # list of per-step tuples
    n_layers = len(cross_attns[0])                   # number of decoder layers
    last_layer_idx = n_layers - 1

    # Stack across decode steps: (tgt_len, src_len)
    attn_per_step = []
    for step_attns in cross_attns:
        # step_attns[layer]: (batch, heads, 1, src_len)
        layer_attn = step_attns[last_layer_idx]      # last layer
        # Average over heads, squeeze batch and step dims
        avg_head = layer_attn[0].mean(dim=0).squeeze(0)   # (src_len,)
        attn_per_step.append(avg_head.numpy())
    attn_matrix = np.array(attn_per_step)            # (tgt_len, src_len)

    # Token labels
    src_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tgt_tokens = tokenizer.convert_ids_to_tokens(tgt_tokens_ids)

    # Trim padding and special tokens for cleaner plot
    src_clean = [t for t in src_tokens if t not in ("</s>", "<pad>")]
    tgt_clean = [t for t in tgt_tokens if t not in ("</s>", "<pad>")]
    attn_plot = attn_matrix[:len(tgt_clean), :len(src_clean)]

    # ── Plot ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(8, len(src_clean)*0.7),
                                    max(6, len(tgt_clean)*0.5)))
    sns.heatmap(
        attn_plot,
        xticklabels=src_clean,
        yticklabels=tgt_clean,
        cmap="YlOrRd",
        linewidths=0.3,
        annot=(attn_plot.shape[0] * attn_plot.shape[1] <= 100),
        fmt=".2f",
        ax=ax,
        vmin=0,
    )
    ax.set_xlabel("Source tokens (English)", labelpad=10)
    ax.set_ylabel("Target tokens (Hindi)", labelpad=10)
    ax.set_title(
        f"Cross-Attention Heatmap — Last Decoder Layer (averaged over heads)\n"
        f"EN: \"{source_text}\"\n"
        f"HI: \"{tgt_text}\"",
        fontsize=10, pad=14,
    )
    plt.xticks(rotation=40, ha="right", fontsize=9)
    plt.yticks(rotation=0,  fontsize=9)
    plt.tight_layout()
    plt.savefig("../outputs/figures/04_bonus_attention_heatmap.png",
                dpi=130, bbox_inches="tight")
    plt.show()

    # Interpret alignment
    print("\nAlignment analysis:")
    print(f"  Source length : {len(src_clean)} tokens")
    print(f"  Target length : {len(tgt_clean)} tokens")
    ratio = len(tgt_clean) / max(len(src_clean), 1)
    print(f"  Length ratio  : {ratio:.2f}  (Hindi tends to be slightly longer than English)")

    peak_cols = attn_plot.argmax(axis=1)
    is_monotone = all(
        peak_cols[i] <= peak_cols[i+1] + 1
        for i in range(len(peak_cols)-1)
    )
    print(f"  Roughly monotone alignment: {is_monotone}")
    print()
    print("  Look for the verb column ('explained') — in Hindi it moves to the")
    print("  end of the sentence (SOV). If the heatmap shows that the Hindi")
    print("  final verb token attends strongly to 'explained', the model has")
    print("  learned SOV reordering purely from parallel data.")

except ImportError as e:
    print(f"Missing dependency: {e}")
    print("Run: pip install transformers sentencepiece torch")
except Exception as e:
    print(f"Error: {e}")
    import traceback; traceback.print_exc()


---
## ⭐ Bonus — Failure Mode: Proverb & Idiom Translation

> **Skip if time is short.** This cell shows where neural MT fundamentally
> fails: idiomatic and proverbial language, where literal translation produces
> fluent but semantically wrong output.

### Background
Neural MT (the neural MT chapter) achieves high BLEU scores on literal
prose. But BLEU measures n-gram overlap with a reference — it does *not*
measure whether the *meaning* transferred correctly.

Proverbs are the hardest test:
- The surface form has high word-overlap with a wrong translation
- The actual meaning requires pragmatic/cultural knowledge
- Back-translation gives confident-looking but wrong output

Classic examples:
| Hindi proverb | Literal translation | Actual meaning |
|--------------|--------------------|-|
| नाच न जाने आँगन टेढ़ा | Can't dance, blames the crooked courtyard | A bad workman blames his tools |
| अब पछताए क्या होत जब चिड़िया चुग गई खेत | What use repenting when the bird has eaten the field | No point crying over spilt milk |


In [ ]:
# ⭐ BONUS — Failure mode: proverb / idiom translation
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from utils.sarvam_helpers import load_client, reset_demo_counters, translate, chat_complete

reset_demo_counters()
client = load_client()

proverbs = [
    {
        "text":    "नाच न जाने आँगन टेढ़ा।",
        "lang":    "hi-IN",
        "literal": "Can't dance, blames the crooked courtyard.",
        "actual":  "A bad workman blames his tools.",
    },
    {
        "text":    "अब पछताए क्या होत जब चिड़िया चुग गई खेत।",
        "lang":    "hi-IN",
        "literal": "What use repenting when the bird has eaten the field.",
        "actual":  "No use crying over spilt milk.",
    },
    {
        "text":    "காற்றுள்ளபோதே தூற்றிக்கொள்.",
        "lang":    "ta-IN",
        "literal": "Winnow while the wind blows.",
        "actual":  "Strike while the iron is hot. (seize the opportunity)",
    },
]

print("FAILURE MODE: Proverb & Idiom Translation")
print("="*65)
print("Neural MT produces fluent output — but does it preserve meaning?\n")

for p in proverbs:
    print(f"Proverb : {p['text']}")
    print(f"Expected: \"{p['actual']}\"")
    try:
        mt_result = translate(client, p["text"], src=p["lang"], tgt="en-IN",
                              mode="formal")
        print(f"MT output: \"{mt_result}\"")
        # Judge whether meaning was preserved
        meaning_ok = any(kw in mt_result.lower()
                         for kw in p["actual"].lower().split()
                         if len(kw) > 4)
        print(f"Meaning preserved: {'✓ probably yes' if meaning_ok else '✗ likely no — literal translation'}")
    except Exception as e:
        print(f"MT error: {e}")

    # Ask Sarvam-M to explain the proverb
    try:
        msg = [{"role": "user",
                "content": f'What does this proverb mean, and what is its '
                           f'English equivalent? Proverb: "{p["text"]}"  '
                           f'Answer in 2 sentences.'}]
        explanation = chat_complete(client, msg)
        if "<think>" in explanation:
            explanation = explanation.split("</think>")[-1].strip()
        print(f"Sarvam-M explanation: {explanation[:300]}")
    except Exception as e:
        print(f"Sarvam-M error: {e}")
    print()

print("Key insight:")
print("  MT systems score well on BLEU because the literal translation shares")
print("  many n-grams with any reference that also uses the words 'dance',")
print("  'courtyard', 'blame' etc. But the meaning is entirely wrong.")
print("  This is why human evaluation and meaning-preserving metrics (e.g.")
print("  COMET, which uses semantic embeddings) are essential for idiomatic text.")


---
## ⭐ Bonus: Real Attention Heatmap from a Neural MT Model
> **Skip if short on time.** This cell loads `Helsinki-NLP/opus-mt-en-hi` (~300 MB) and visualises the actual cross-attention weights the decoder uses when translating English → Hindi. You will see, concretely, which source tokens each target token attends to.

### What the attention mechanism does (transformer architecture section)
In a transformer encoder-decoder (the transformer architecture section), the decoder generates each target token by computing **cross-attention** over all encoder hidden states:

```
score(query_i, key_j) = (Q_i · K_j) / √d_k
attention_weight(i,j) = softmax(score)_j
output_i = Σ_j attention_weight(i,j) · V_j
```

The attention weight matrix has shape `[target_length × source_length]`. Each row is a probability distribution over source tokens — it answers: *"when generating target token i, how much did I look at each source token?"*

### What to look for in the heatmap
- **Diagonal pattern** → roughly monotonic alignment (common in closely related language pairs)
- **Off-diagonal bright cells** → long-distance reordering (expected for EN→HI: verb moves from position 3 to final position)
- **Diffuse rows** → decoder is uncertain which source token to attend to (often at punctuation or function words)
- **Sharp rows** → content words like nouns and verbs tend to have sharp, focused attention

### Why this is especially interesting for SOV languages
English "Ram eats an apple" is SVO. Hindi "राम एक सेब खाता है" is SOV — the verb is at the end. Watch the attention heatmap: the Hindi verb token should attend strongly to the English verb token at position 2, even though it's generated last. This is the attention mechanism solving the reordering problem that IBM models (the statistical IBM models section) required explicit distortion parameters to handle.

### ⭐ BONUS — Real Cross-Attention Heatmap from Helsinki-NLP opus-mt-en-hi

In [ ]:
# Requires: pip install transformers sentencepiece sacremoses
# Downloads Helsinki-NLP/opus-mt-en-hi (~300 MB, cached after first run)

import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    from transformers import MarianMTModel, MarianTokenizer
    import torch
    HAS_TRANSFORMERS = True
except ImportError:
    print("transformers or torch not installed.")
    print("Run: pip install transformers torch sentencepiece sacremoses")
    HAS_TRANSFORMERS = False

if HAS_TRANSFORMERS:
    MODEL_NAME = "Helsinki-NLP/opus-mt-en-hi"
    print(f"Loading {MODEL_NAME}...")
    print("(~300 MB on first run, then cached)")
    
    tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
    model = MarianMTModel.from_pretrained(MODEL_NAME, output_attentions=True)
    model.eval()
    print("  ✓ Model loaded\n")

    # ── Sentences chosen to show reordering ───────────────────────────────
    test_sentences = [
        "Ram eats an apple in school.",
        "The teacher explains language processing.",
        "Students come to the computer science department.",
    ]

    for sent_idx, src_text in enumerate(test_sentences):
        print(f"{'='*60}")
        print(f"Source: {src_text}")

        # ── Tokenise & translate ───────────────────────────────────────────
        inputs = tokenizer(src_text, return_tensors="pt", padding=True)
        src_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                output_attentions=True,
                return_dict_in_generate=True,
                num_beams=1,           # greedy — easier to inspect
                max_new_tokens=50,
            )

        # Decode translation
        translation = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
        print(f"Translation: {translation}")

        # Decode target tokens
        tgt_ids = outputs.sequences[0]
        tgt_tokens = tokenizer.convert_ids_to_tokens(tgt_ids)
        tgt_tokens = [t for t in tgt_tokens if t not in tokenizer.all_special_tokens]

        # ── Extract cross-attention weights ───────────────────────────────
        # outputs.cross_attentions: tuple of tuples
        # Shape: [decoding_step][layer_idx] → tensor [batch, heads, 1, src_len]
        # We average across heads and take the last decoder layer
        n_steps = len(outputs.cross_attentions)
        n_layers = len(outputs.cross_attentions[0])
        src_len = inputs["input_ids"].shape[1]
        tgt_len = min(n_steps, len(tgt_tokens))

        # Build attention matrix: [tgt_len, src_len]
        attn_matrix = np.zeros((tgt_len, src_len))
        for step in range(tgt_len):
            # Last decoder layer, average over heads
            layer_attn = outputs.cross_attentions[step][-1]  # [batch, heads, 1, src_len]
            avg_attn = layer_attn[0].mean(dim=0)[0].numpy()  # [src_len]
            avg_attn = avg_attn[:src_len]
            attn_matrix[step] = avg_attn / (avg_attn.sum() + 1e-9)

        # ── Plot ───────────────────────────────────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, max(4, tgt_len * 0.5 + 2)))

        # Left: heatmap
        ax = axes[0]
        clean_src = [t.replace("▁", " ").strip() for t in src_tokens]
        clean_tgt = [t.replace("▁", " ").strip() for t in tgt_tokens[:tgt_len]]
        
        im = ax.imshow(attn_matrix, aspect="auto", cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(clean_src)))
        ax.set_xticklabels(clean_src, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(len(clean_tgt)))
        ax.set_yticklabels(clean_tgt, fontsize=9)
        ax.set_xlabel("Source (English) tokens")
        ax.set_ylabel("Target (Hindi) tokens")
        ax.set_title(f"Cross-Attention Weights\n(last decoder layer, avg over heads)\n\"{src_text[:40]}\"")
        plt.colorbar(im, ax=ax, shrink=0.8)

        # Right: annotated alignment with arrows for top-attention pairs
        ax2 = axes[1]
        ax2.axis("off")
        
        lines = [f"Sentence {sent_idx+1} alignment analysis:\n",
                 f"Source:      {src_text}\n",
                 f"Translation: {translation}\n\n",
                 "Top attention pairs (target → source):\n"]
        
        for tgt_i, tgt_tok in enumerate(clean_tgt):
            if tgt_i >= attn_matrix.shape[0]:
                break
            top_src_idx = np.argmax(attn_matrix[tgt_i])
            top_src_tok = clean_src[top_src_idx] if top_src_idx < len(clean_src) else "?"
            weight = attn_matrix[tgt_i, top_src_idx]
            if weight > 0.15:  # only show meaningful alignments
                lines.append(f"  '{tgt_tok}' ← '{top_src_tok}' ({weight:.2f})\n")
        
        ax2.text(0.05, 0.95, "".join(lines), transform=ax2.transAxes,
                fontsize=9, va="top", fontfamily="monospace",
                bbox=dict(boxstyle="round", facecolor="#e8f4fd", alpha=0.9))

        plt.tight_layout()
        save_name = f"04_bonus_attention_{sent_idx+1}.png"
        plt.savefig(f"../outputs/figures/{save_name}", dpi=120, bbox_inches="tight")
        plt.show()
        
        # Reordering analysis
        print(f"\nReordering insight:")
        print(f"  English verb position: {[i for i, t in enumerate(clean_src) if t.lower() in ['eats', 'explains', 'come', 'comes']]}")
        print(f"  Hindi tokens (end of sentence tend to be verbs in SOV):")
        if len(clean_tgt) >= 2:
            print(f"  Last 2 Hindi tokens: {clean_tgt[-2:]}")
        print()

---
## Krutrim Comparison — Translation: 4-Way BLEU Benchmark

> **Requires:** `KRUTRIM_CLOUD_API_KEY` in `.env`
> **Requires:** `nltk` (already installed)

### Background: Four Translation Systems, One Benchmark

The neural MT chapter distinguishes **dedicated MT models** (trained on
parallel corpora with seq2seq objectives) from **LLMs used as zero-shot
translators** (prompted to translate). Both approaches are in widespread use;
this cell lets you measure the gap.

| System | Architecture | Training objective |
|--------|-------------|-------------------|
| **Sarvam Mayura v1** | Dedicated MT (Sarvam) | Parallel EN-Indic, 4 style modes |
| **Sarvam-Translate v1** | Dedicated MT (Sarvam) | Parallel EN-Indic, alternative model |
| **KrutrimTranslate** | Dedicated MT (Krutrim, IndicTrans2-based) | 9 Indic languages |
| **Krutrim-spectre-v2** | LLM zero-shot | Chat fine-tuning, no MT-specific training |

We run all four on the same sentence pairs and compute sentence-BLEU against
human reference translations. The gap between dedicated MT and LLM zero-shot
is the empirical answer to: *"Do you need a specialised MT model, or does a
general Indic LLM do just as well?"*


In [ ]:
# [DISABLED] Krutrim API key unavailable — uncomment if you have a key
#
# # N
# o
# t
# e
# b
# o
# o
# k
 # 0
# 4
 # —
 # 4
# -
# w
# a
# y
 # t
# r
# a
# n
# s
# l
# a
# t
# i
# o
# n
 # B
# L
# E
# U
# :
 # S
# a
# r
# v
# a
# m
 # (
# x
# 2
# )
 # v
# s
 # K
# r
# u
# t
# r
# i
# m
 # (
# x
# 2
# )

# i
# m
# p
# o
# r
# t
 # s
# y
# s
# ,
 # o
# s

# s
# y
# s
# .
# p
# a
# t
# h
# .
# i
# n
# s
# e
# r
# t
# (
# 0
# ,
 # o
# s
# .
# p
# a
# t
# h
# .
# a
# b
# s
# p
# a
# t
# h
# (
# '
# .
# .
# '
# )
# )


# t
# r
# y
# :

    # f
# r
# o
# m
 # u
# t
# i
# l
# s
# .
# k
# r
# u
# t
# r
# i
# m
# _
# h
# e
# l
# p
# e
# r
# s
 # i
# m
# p
# o
# r
# t
 # (

        # l
# o
# a
# d
# _
# o
# p
# e
# n
# a
# i
# _
# c
# l
# i
# e
# n
# t
# ,
 # l
# o
# a
# d
# _
# k
# r
# u
# t
# r
# i
# m
# _
# c
# l
# i
# e
# n
# t
# ,

        # k
# r
# u
# t
# r
# i
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# ,
 # k
# r
# u
# t
# r
# i
# m
# _
# c
# h
# a
# t
# ,

        # S
# A
# R
# V
# A
# M
# _
# T
# O
# _
# K
# R
# U
# T
# R
# I
# M
# _
# L
# A
# N
# G
# ,
 # K
# R
# U
# T
# R
# I
# M
# _
# L
# A
# N
# G
# _
# N
# A
# M
# E
# S
# ,

    # )

    # f
# r
# o
# m
 # u
# t
# i
# l
# s
# .
# s
# a
# r
# v
# a
# m
# _
# h
# e
# l
# p
# e
# r
# s
 # i
# m
# p
# o
# r
# t
 # (

        # l
# o
# a
# d
# _
# c
# l
# i
# e
# n
# t
# ,
 # t
# r
# a
# n
# s
# l
# a
# t
# e
 # a
# s
 # s
# a
# r
# v
# a
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# ,

        # r
# e
# s
# e
# t
# _
# d
# e
# m
# o
# _
# c
# o
# u
# n
# t
# e
# r
# s
# ,
 # p
# l
# o
# t
# _
# b
# l
# e
# u
# _
# c
# o
# m
# p
# a
# r
# i
# s
# o
# n
# ,

    # )

    # f
# r
# o
# m
 # d
# a
# t
# a
# .
# s
# a
# m
# p
# l
# e
# _
# t
# e
# x
# t
# s
 # i
# m
# p
# o
# r
# t
 # L
# A
# N
# G
# U
# A
# G
# E
# _
# N
# A
# M
# E
# S

    # i
# m
# p
# o
# r
# t
 # n
# l
# t
# k
# ,
 # n
# u
# m
# p
# y
 # a
# s
 # n
# p
# ,
 # p
# a
# n
# d
# a
# s
 # a
# s
 # p
# d

    # f
# r
# o
# m
 # n
# l
# t
# k
# .
# t
# r
# a
# n
# s
# l
# a
# t
# e
# .
# b
# l
# e
# u
# _
# s
# c
# o
# r
# e
 # i
# m
# p
# o
# r
# t
 # s
# e
# n
# t
# e
# n
# c
# e
# _
# b
# l
# e
# u
# ,
 # S
# m
# o
# o
# t
# h
# i
# n
# g
# F
# u
# n
# c
# t
# i
# o
# n

    # i
# m
# p
# o
# r
# t
 # m
# a
# t
# p
# l
# o
# t
# l
# i
# b
# .
# p
# y
# p
# l
# o
# t
 # a
# s
 # p
# l
# t
# ,
 # s
# e
# a
# b
# o
# r
# n
 # a
# s
 # s
# n
# s


    # n
# l
# t
# k
# .
# d
# o
# w
# n
# l
# o
# a
# d
# (
# "
# p
# u
# n
# k
# t
# "
# ,
 # q
# u
# i
# e
# t
# =
# T
# r
# u
# e
# )

    # n
# l
# t
# k
# .
# d
# o
# w
# n
# l
# o
# a
# d
# (
# "
# p
# u
# n
# k
# t
# _
# t
# a
# b
# "
# ,
 # q
# u
# i
# e
# t
# =
# T
# r
# u
# e
# )

    # s
# m
# o
# o
# t
# h
# e
# r
 # =
 # S
# m
# o
# o
# t
# h
# i
# n
# g
# F
# u
# n
# c
# t
# i
# o
# n
# (
# )
# .
# m
# e
# t
# h
# o
# d
# 1


    # r
# e
# s
# e
# t
# _
# d
# e
# m
# o
# _
# c
# o
# u
# n
# t
# e
# r
# s
# (
# )

    # s
# a
# r
# v
# a
# m
          # =
 # l
# o
# a
# d
# _
# c
# l
# i
# e
# n
# t
# (
# )

    # k
# r
# u
# t
# r
# i
# m
# _
# o
# a
# i
     # =
 # l
# o
# a
# d
# _
# o
# p
# e
# n
# a
# i
# _
# c
# l
# i
# e
# n
# t
# (
# )

    # t
# r
# y
# :

        # k
# r
# u
# t
# r
# i
# m
# _
# n
# a
# t
 # =
 # l
# o
# a
# d
# _
# k
# r
# u
# t
# r
# i
# m
# _
# c
# l
# i
# e
# n
# t
# (
# )

        # u
# s
# e
# _
# k
# r
# u
# t
# r
# i
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
 # =
 # T
# r
# u
# e

    # e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
# :

        # k
# r
# u
# t
# r
# i
# m
# _
# n
# a
# t
 # =
 # N
# o
# n
# e

        # u
# s
# e
# _
# k
# r
# u
# t
# r
# i
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
 # =
 # F
# a
# l
# s
# e

        # p
# r
# i
# n
# t
# (
# "
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # S
# D
# K
 # u
# n
# a
# v
# a
# i
# l
# a
# b
# l
# e
# ;
 # w
# i
# l
# l
 # u
# s
# e
 # L
# L
# M
 # z
# e
# r
# o
# -
# s
# h
# o
# t
 # o
# n
# l
# y
# .
# "
# )


    # # S
# e
# n
# t
# e
# n
# c
# e
 # p
# a
# i
# r
# s
# :
 # (
# l
# a
# n
# g
# _
# s
# a
# r
# v
# a
# m
# ,
 # s
# o
# u
# r
# c
# e
# ,
 # h
# u
# m
# a
# n
# _
# r
# e
# f
# e
# r
# e
# n
# c
# e
# )

    # p
# a
# i
# r
# s
 # =
 # [

        # (
# "
# h
# i
# -
# I
# N
# "
# ,
 # "
# र
# ा
# म
 # स
# ्
# क
# ू
# ल
 # ज
# ा
# त
# ा
 # ह
# ै
# ।
# "
# ,
            # "
# R
# a
# m
 # g
# o
# e
# s
 # t
# o
 # s
# c
# h
# o
# o
# l
# .
# "
# )
# ,

        # (
# "
# h
# i
# -
# I
# N
# "
# ,
 # "
# आ
# ज
 # म
# ौ
# स
# म
 # ब
# ह
# ु
# त
 # अ
# च
# ्
# छ
# ा
 # ह
# ै
# ।
# "
# ,
          # "
# T
# h
# e
 # w
# e
# a
# t
# h
# e
# r
 # i
# s
 # v
# e
# r
# y
 # g
# o
# o
# d
 # t
# o
# d
# a
# y
# .
# "
# )
# ,

        # (
# "
# t
# a
# -
# I
# N
# "
# ,
 # "
# அ
# வ
# ர
# ்
 # ந
# ன
# ்
# ற
# ா
# க
 # ப
# ா
# ட
# ு
# க
# ி
# ற
# ா
# ர
# ்
# .
# "
# ,
         # "
# H
# e
 # s
# i
# n
# g
# s
 # w
# e
# l
# l
# .
# "
# )
# ,

        # (
# "
# b
# n
# -
# I
# N
# "
# ,
 # "
# আ
# ম
# ি
 # ব
# া
# ং
# ল
# া
 # ভ
# া
# ষ
# া
 # ভ
# া
# ল
# ো
# ব
# া
# স
# ি
# ।
# "
# ,
        # "
# I
 # l
# o
# v
# e
 # t
# h
# e
 # B
# e
# n
# g
# a
# l
# i
 # l
# a
# n
# g
# u
# a
# g
# e
# .
# "
# )
# ,

        # (
# "
# t
# e
# -
# I
# N
# "
# ,
 # "
# భ
# ా
# ష
# ా
 # ప
# ్
# ర
# ా
# స
# ె
# స
# ి
# ం
# గ
# ్
 # మ
# ు
# ఖ
# ్
# య
# మ
# ై
# న
# ద
# ి
# .
# "
# ,
    # "
# L
# a
# n
# g
# u
# a
# g
# e
 # p
# r
# o
# c
# e
# s
# s
# i
# n
# g
 # i
# s
 # i
# m
# p
# o
# r
# t
# a
# n
# t
# .
# "
# )
# ,

    # ]


    # s
# y
# s
# t
# e
# m
# s
 # =
 # [
# "
# M
# a
# y
# u
# r
# a
 # v
# 1
# "
# ,
 # "
# S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1
# "
# ,

               # "
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# "
# ,
 # "
# K
# r
# u
# t
# r
# i
# m
# -
# L
# L
# M
 # (
# 0
# -
# s
# h
# o
# t
# )
# "
# ]

    # b
# l
# e
# u
# _
# s
# c
# o
# r
# e
# s
# :
 # d
# i
# c
# t
# [
# s
# t
# r
# ,
 # d
# i
# c
# t
# [
# s
# t
# r
# ,
 # l
# i
# s
# t
# ]
# ]
 # =
 # {
# s
# :
 # {
# }
 # f
# o
# r
 # s
 # i
# n
 # s
# y
# s
# t
# e
# m
# s
# }


    # p
# r
# i
# n
# t
# (
# f
# "
# {
# '
# S
# o
# u
# r
# c
# e
# '
# :
# <
# 3
# 5
# }
 # {
# '
# M
# a
# y
# u
# r
# a
# '
# :
# >
# 8
# }
 # {
# '
# S
# a
# r
# v
# T
# r
# '
# :
# >
# 8
# }
 # {
# '
# K
# r
# u
# T
# r
# '
# :
# >
# 8
# }
 # {
# '
# K
# r
# u
# L
# L
# M
# '
# :
# >
# 9
# }
# "
# )

    # p
# r
# i
# n
# t
# (
# "
# -
# "
# *
# 7
# 2
# )


    # f
# o
# r
 # l
# a
# n
# g
# ,
 # s
# r
# c
# ,
 # r
# e
# f
 # i
# n
 # p
# a
# i
# r
# s
# :

        # l
# a
# n
# g
# _
# k
 # =
 # S
# A
# R
# V
# A
# M
# _
# T
# O
# _
# K
# R
# U
# T
# R
# I
# M
# _
# L
# A
# N
# G
# .
# g
# e
# t
# (
# l
# a
# n
# g
# ,
 # l
# a
# n
# g
# .
# s
# p
# l
# i
# t
# (
# "
# -
# "
# )
# [
# 0
# ]
# )

        # r
# e
# f
# _
# t
# o
# k
 # =
 # r
# e
# f
# .
# l
# o
# w
# e
# r
# (
# )
# .
# s
# p
# l
# i
# t
# (
# )


        # r
# o
# w
 # =
 # {
# }


        # # 1
# .
 # M
# a
# y
# u
# r
# a
 # v
# 1

        # t
# r
# y
# :

            # t
 # =
 # s
# a
# r
# v
# a
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# (
# s
# a
# r
# v
# a
# m
# ,
 # s
# r
# c
# ,
 # s
# r
# c
# =
# l
# a
# n
# g
# ,
 # t
# g
# t
# =
# "
# e
# n
# -
# I
# N
# "
# ,

                                 # m
# o
# d
# e
# l
# =
# "
# m
# a
# y
# u
# r
# a
# :
# v
# 1
# "
# )

            # r
# o
# w
# [
# "
# M
# a
# y
# u
# r
# a
 # v
# 1
# "
# ]
 # =
 # s
# e
# n
# t
# e
# n
# c
# e
# _
# b
# l
# e
# u
# (
# [
# r
# e
# f
# _
# t
# o
# k
# ]
# ,
 # t
# .
# l
# o
# w
# e
# r
# (
# )
# .
# s
# p
# l
# i
# t
# (
# )
# ,

                                             # s
# m
# o
# o
# t
# h
# i
# n
# g
# _
# f
# u
# n
# c
# t
# i
# o
# n
# =
# s
# m
# o
# o
# t
# h
# e
# r
# )

        # e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
 # a
# s
 # e
# :

            # r
# o
# w
# [
# "
# M
# a
# y
# u
# r
# a
 # v
# 1
# "
# ]
 # =
 # 0
# .
# 0


        # # 2
# .
 # S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1

        # t
# r
# y
# :

            # t
 # =
 # s
# a
# r
# v
# a
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# (
# s
# a
# r
# v
# a
# m
# ,
 # s
# r
# c
# ,
 # s
# r
# c
# =
# l
# a
# n
# g
# ,
 # t
# g
# t
# =
# "
# e
# n
# -
# I
# N
# "
# ,

                                 # m
# o
# d
# e
# l
# =
# "
# s
# a
# r
# v
# a
# m
# -
# t
# r
# a
# n
# s
# l
# a
# t
# e
# :
# v
# 1
# "
# )

            # r
# o
# w
# [
# "
# S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1
# "
# ]
 # =
 # s
# e
# n
# t
# e
# n
# c
# e
# _
# b
# l
# e
# u
# (

                # [
# r
# e
# f
# _
# t
# o
# k
# ]
# ,
 # t
# .
# l
# o
# w
# e
# r
# (
# )
# .
# s
# p
# l
# i
# t
# (
# )
# ,
 # s
# m
# o
# o
# t
# h
# i
# n
# g
# _
# f
# u
# n
# c
# t
# i
# o
# n
# =
# s
# m
# o
# o
# t
# h
# e
# r
# )

        # e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
 # a
# s
 # e
# :

            # r
# o
# w
# [
# "
# S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1
# "
# ]
 # =
 # 0
# .
# 0


        # # 3
# .
 # K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e

        # i
# f
 # u
# s
# e
# _
# k
# r
# u
# t
# r
# i
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# :

            # t
# r
# y
# :

                # t
 # =
 # k
# r
# u
# t
# r
# i
# m
# _
# t
# r
# a
# n
# s
# l
# a
# t
# e
# (
# k
# r
# u
# t
# r
# i
# m
# _
# n
# a
# t
# ,
 # s
# r
# c
# ,
 # s
# r
# c
# =
# l
# a
# n
# g
# _
# k
# ,
 # t
# g
# t
# =
# "
# e
# n
# "
# )

                # r
# o
# w
# [
# "
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# "
# ]
 # =
 # s
# e
# n
# t
# e
# n
# c
# e
# _
# b
# l
# e
# u
# (

                    # [
# r
# e
# f
# _
# t
# o
# k
# ]
# ,
 # t
# .
# l
# o
# w
# e
# r
# (
# )
# .
# s
# p
# l
# i
# t
# (
# )
# ,
 # s
# m
# o
# o
# t
# h
# i
# n
# g
# _
# f
# u
# n
# c
# t
# i
# o
# n
# =
# s
# m
# o
# o
# t
# h
# e
# r
# )

            # e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
# :

                # r
# o
# w
# [
# "
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# "
# ]
 # =
 # 0
# .
# 0

        # e
# l
# s
# e
# :

            # r
# o
# w
# [
# "
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# "
# ]
 # =
 # N
# o
# n
# e


        # # 4
# .
 # K
# r
# u
# t
# r
# i
# m
 # L
# L
# M
 # z
# e
# r
# o
# -
# s
# h
# o
# t

        # t
# r
# y
# :

            # s
# r
# c
# _
# n
# a
# m
# e
 # =
 # K
# R
# U
# T
# R
# I
# M
# _
# L
# A
# N
# G
# _
# N
# A
# M
# E
# S
# .
# g
# e
# t
# (
# l
# a
# n
# g
# _
# k
# ,
 # l
# a
# n
# g
# _
# k
# )

            # p
# r
# o
# m
# p
# t
 # =
 # (
# f
# "
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # t
# h
# i
# s
 # {
# s
# r
# c
# _
# n
# a
# m
# e
# }
 # s
# e
# n
# t
# e
# n
# c
# e
 # t
# o
 # E
# n
# g
# l
# i
# s
# h
# .
 # "

                      # f
# "
# O
# u
# t
# p
# u
# t
 # o
# n
# l
# y
 # t
# h
# e
 # t
# r
# a
# n
# s
# l
# a
# t
# i
# o
# n
# .


# {
# s
# r
# c
# }
# "
# )

            # t
 # =
 # k
# r
# u
# t
# r
# i
# m
# _
# c
# h
# a
# t
# (
# k
# r
# u
# t
# r
# i
# m
# _
# o
# a
# i
# ,

                             # [
# {
# "
# r
# o
# l
# e
# "
# :
 # "
# u
# s
# e
# r
# "
# ,
 # "
# c
# o
# n
# t
# e
# n
# t
# "
# :
 # p
# r
# o
# m
# p
# t
# }
# ]
# ,

                             # t
# e
# m
# p
# e
# r
# a
# t
# u
# r
# e
# =
# 0
# .
# 1
# ,
 # m
# a
# x
# _
# t
# o
# k
# e
# n
# s
# =
# 1
# 2
# 8
# )

            # r
# o
# w
# [
# "
# K
# r
# u
# t
# r
# i
# m
# -
# L
# L
# M
 # (
# 0
# -
# s
# h
# o
# t
# )
# "
# ]
 # =
 # s
# e
# n
# t
# e
# n
# c
# e
# _
# b
# l
# e
# u
# (

                # [
# r
# e
# f
# _
# t
# o
# k
# ]
# ,
 # t
# .
# l
# o
# w
# e
# r
# (
# )
# .
# s
# p
# l
# i
# t
# (
# )
# ,
 # s
# m
# o
# o
# t
# h
# i
# n
# g
# _
# f
# u
# n
# c
# t
# i
# o
# n
# =
# s
# m
# o
# o
# t
# h
# e
# r
# )

        # e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
 # a
# s
 # e
# :

            # r
# o
# w
# [
# "
# K
# r
# u
# t
# r
# i
# m
# -
# L
# L
# M
 # (
# 0
# -
# s
# h
# o
# t
# )
# "
# ]
 # =
 # 0
# .
# 0


        # f
# o
# r
 # s
 # i
# n
 # s
# y
# s
# t
# e
# m
# s
# :

            # b
# l
# e
# u
# _
# s
# c
# o
# r
# e
# s
# [
# s
# ]
# .
# s
# e
# t
# d
# e
# f
# a
# u
# l
# t
# (
# l
# a
# n
# g
# ,
 # [
# ]
# )
# .
# a
# p
# p
# e
# n
# d
# (
# r
# o
# w
# .
# g
# e
# t
# (
# s
# ,
 # 0
# .
# 0
# )
 # o
# r
 # 0
# .
# 0
# )


        # p
# r
# i
# n
# t
# (
# f
# "
# {
# s
# r
# c
# [
# :
# 3
# 3
# ]
# :
# <
# 3
# 5
# }
 # "

              # f
# "
# {
# r
# o
# w
# .
# g
# e
# t
# (
# '
# M
# a
# y
# u
# r
# a
 # v
# 1
# '
# ,
 # 0
# )
# :
# >
# 8
# .
# 2
# f
# }
 # "

              # f
# "
# {
# r
# o
# w
# .
# g
# e
# t
# (
# '
# S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1
# '
# ,
 # 0
# )
# :
# >
# 8
# .
# 2
# f
# }
 # "

              # f
# "
# {
# r
# o
# w
# .
# g
# e
# t
# (
# '
# K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# '
# )
 # o
# r
 # 0
# :
# >
# 8
# .
# 2
# f
# }
 # "

              # f
# "
# {
# r
# o
# w
# .
# g
# e
# t
# (
# '
# K
# r
# u
# t
# r
# i
# m
# -
# L
# L
# M
 # (
# 0
# -
# s
# h
# o
# t
# )
# '
# ,
 # 0
# )
# :
# >
# 9
# .
# 2
# f
# }
# "
# )


    # # A
# v
# e
# r
# a
# g
# e
 # p
# e
# r
 # l
# a
# n
# g
# u
# a
# g
# e

    # a
# v
# g
 # =
 # {
# s
# :
 # {
# l
# :
 # f
# l
# o
# a
# t
# (
# n
# p
# .
# m
# e
# a
# n
# (
# v
# )
# )
 # f
# o
# r
 # l
# ,
 # v
 # i
# n
 # l
# d
# .
# i
# t
# e
# m
# s
# (
# )
# }

           # f
# o
# r
 # s
# ,
 # l
# d
 # i
# n
 # b
# l
# e
# u
# _
# s
# c
# o
# r
# e
# s
# .
# i
# t
# e
# m
# s
# (
# )
# }


    # p
# r
# i
# n
# t
# (
# "

# A
# v
# e
# r
# a
# g
# e
 # B
# L
# E
# U
 # b
# y
 # l
# a
# n
# g
# u
# a
# g
# e
# :
# "
# )

    # d
# f
 # =
 # p
# d
# .
# D
# a
# t
# a
# F
# r
# a
# m
# e
# (
# a
# v
# g
# )
# .
# T

    # p
# r
# i
# n
# t
# (
# d
# f
# .
# r
# o
# u
# n
# d
# (
# 3
# )
# .
# t
# o
# _
# s
# t
# r
# i
# n
# g
# (
# )
# )


    # # ─
# ─
 # G
# r
# o
# u
# p
# e
# d
 # b
# a
# r
 # c
# h
# a
# r
# t
 # ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─
# ─

    # l
# a
# n
# g
# u
# a
# g
# e
# s
 # =
 # s
# o
# r
# t
# e
# d
# (
# {
# l
 # f
# o
# r
 # l
# d
 # i
# n
 # a
# v
# g
# .
# v
# a
# l
# u
# e
# s
# (
# )
 # f
# o
# r
 # l
 # i
# n
 # l
# d
# }
# )

    # x
 # =
 # n
# p
# .
# a
# r
# a
# n
# g
# e
# (
# l
# e
# n
# (
# l
# a
# n
# g
# u
# a
# g
# e
# s
# )
# )

    # w
 # =
 # 0
# .
# 1
# 8

    # c
# o
# l
# o
# r
# s
 # =
 # [
# "
## F
# F
# 6
# B
# 3
# 5
# "
# ,
 # "
## 4
# E
# C
# D
# C
# 4
# "
# ,
 # "
## F
# F
# 9
# F
# 4
# 3
# "
# ,
 # "
## 4
# 8
# D
# B
# F
# B
# "
# ]


    # f
# i
# g
# ,
 # a
# x
 # =
 # p
# l
# t
# .
# s
# u
# b
# p
# l
# o
# t
# s
# (
# f
# i
# g
# s
# i
# z
# e
# =
# (
# 1
# 3
# ,
 # 5
# )
# )

    # f
# o
# r
 # i
# ,
 # (
# s
# y
# s
# _
# n
# a
# m
# e
# ,
 # c
# o
# l
# o
# r
# )
 # i
# n
 # e
# n
# u
# m
# e
# r
# a
# t
# e
# (
# z
# i
# p
# (
# s
# y
# s
# t
# e
# m
# s
# ,
 # c
# o
# l
# o
# r
# s
# )
# )
# :

        # s
# c
# o
# r
# e
# s
 # =
 # [
# a
# v
# g
# [
# s
# y
# s
# _
# n
# a
# m
# e
# ]
# .
# g
# e
# t
# (
# l
# ,
 # 0
# )
 # f
# o
# r
 # l
 # i
# n
 # l
# a
# n
# g
# u
# a
# g
# e
# s
# ]

        # o
# f
# f
# s
# e
# t
 # =
 # (
# i
 # -
 # l
# e
# n
# (
# s
# y
# s
# t
# e
# m
# s
# )
# /
# 2
 # +
 # 0
# .
# 5
# )
 # *
 # w

        # b
# a
# r
# s
 # =
 # a
# x
# .
# b
# a
# r
# (
# x
 # +
 # o
# f
# f
# s
# e
# t
# ,
 # s
# c
# o
# r
# e
# s
# ,
 # w
# ,
 # l
# a
# b
# e
# l
# =
# s
# y
# s
# _
# n
# a
# m
# e
# ,

                      # c
# o
# l
# o
# r
# =
# c
# o
# l
# o
# r
# ,
 # a
# l
# p
# h
# a
# =
# 0
# .
# 8
# 8
# ,
 # e
# d
# g
# e
# c
# o
# l
# o
# r
# =
# "
# w
# h
# i
# t
# e
# "
# )

        # a
# x
# .
# b
# a
# r
# _
# l
# a
# b
# e
# l
# (
# b
# a
# r
# s
# ,
 # f
# m
# t
# =
# "
# %
# .
# 2
# f
# "
# ,
 # p
# a
# d
# d
# i
# n
# g
# =
# 2
# ,
 # f
# o
# n
# t
# s
# i
# z
# e
# =
# 8
# )


    # a
# x
# .
# s
# e
# t
# _
# x
# t
# i
# c
# k
# s
# (
# x
# )

    # a
# x
# .
# s
# e
# t
# _
# x
# t
# i
# c
# k
# l
# a
# b
# e
# l
# s
# (
# [
# L
# A
# N
# G
# U
# A
# G
# E
# _
# N
# A
# M
# E
# S
# .
# g
# e
# t
# (
# l
# ,
 # l
# )
 # f
# o
# r
 # l
 # i
# n
 # l
# a
# n
# g
# u
# a
# g
# e
# s
# ]
# )

    # a
# x
# .
# s
# e
# t
# _
# y
# l
# a
# b
# e
# l
# (
# "
# S
# e
# n
# t
# e
# n
# c
# e
 # B
# L
# E
# U
 # (
# s
# m
# o
# o
# t
# h
# e
# d
# ,
 # h
# i
# g
# h
# e
# r
 # i
# s
 # b
# e
# t
# t
# e
# r
# )
# "
# )

    # a
# x
# .
# s
# e
# t
# _
# t
# i
# t
# l
# e
# (
# "
# 4
# -
# W
# a
# y
 # T
# r
# a
# n
# s
# l
# a
# t
# i
# o
# n
 # B
# L
# E
# U
# :
 # S
# a
# r
# v
# a
# m
 # M
# a
# y
# u
# r
# a
 # v
# 1
 # &
 # S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # v
# 1

# "

                 # "
# v
# s
 # K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
 # &
 # K
# r
# u
# t
# r
# i
# m
# -
# s
# p
# e
# c
# t
# r
# e
# -
# v
# 2
 # z
# e
# r
# o
# -
# s
# h
# o
# t
  # |
  # -
# >
 # E
# n
# g
# l
# i
# s
# h
# "
# )

    # a
# x
# .
# l
# e
# g
# e
# n
# d
# (
# l
# o
# c
# =
# "
# u
# p
# p
# e
# r
 # r
# i
# g
# h
# t
# "
# ,
 # f
# o
# n
# t
# s
# i
# z
# e
# =
# 9
# )

    # a
# x
# .
# s
# e
# t
# _
# y
# l
# i
# m
# (
# 0
# ,
 # 1
# .
# 0
# )

    # s
# n
# s
# .
# d
# e
# s
# p
# i
# n
# e
# (
# )

    # p
# l
# t
# .
# t
# i
# g
# h
# t
# _
# l
# a
# y
# o
# u
# t
# (
# )

    # p
# l
# t
# .
# s
# a
# v
# e
# f
# i
# g
# (
# "
# .
# .
# /
# o
# u
# t
# p
# u
# t
# s
# /
# f
# i
# g
# u
# r
# e
# s
# /
# 0
# 4
# _
# k
# r
# u
# t
# r
# i
# m
# _
# v
# s
# _
# s
# a
# r
# v
# a
# m
# _
# b
# l
# e
# u
# .
# p
# n
# g
# "
# ,

                # d
# p
# i
# =
# 1
# 2
# 0
# ,
 # b
# b
# o
# x
# _
# i
# n
# c
# h
# e
# s
# =
# "
# t
# i
# g
# h
# t
# "
# )

    # p
# l
# t
# .
# s
# h
# o
# w
# (
# )


    # p
# r
# i
# n
# t
# (
# "

# K
# e
# y
 # q
# u
# e
# s
# t
# i
# o
# n
# :
 # h
# o
# w
 # l
# a
# r
# g
# e
 # i
# s
 # t
# h
# e
 # g
# a
# p
 # b
# e
# t
# w
# e
# e
# n
 # d
# e
# d
# i
# c
# a
# t
# e
# d
 # M
# T
 # m
# o
# d
# e
# l
# s
# "
# )

    # p
# r
# i
# n
# t
# (
# "
# (
# M
# a
# y
# u
# r
# a
# ,
 # S
# a
# r
# v
# a
# m
# -
# T
# r
# a
# n
# s
# l
# a
# t
# e
# ,
 # K
# r
# u
# t
# r
# i
# m
# T
# r
# a
# n
# s
# l
# a
# t
# e
# )
 # a
# n
# d
 # t
# h
# e
 # L
# L
# M
 # z
# e
# r
# o
# -
# s
# h
# o
# t
# "
# )

    # p
# r
# i
# n
# t
# (
# "
# (
# K
# r
# u
# t
# r
# i
# m
# -
# s
# p
# e
# c
# t
# r
# e
# -
# v
# 2
# )
# ?
 # A
 # s
# m
# a
# l
# l
 # g
# a
# p
 # m
# e
# a
# n
# s
 # L
# L
# M
# s
 # c
# a
# n
 # s
# u
# b
# s
# t
# i
# t
# u
# t
# e
 # f
# o
# r
 # M
# T
# .
# "
# )

    # p
# r
# i
# n
# t
# (
# "
# A
 # l
# a
# r
# g
# e
 # g
# a
# p
 # m
# e
# a
# n
# s
 # d
# e
# d
# i
# c
# a
# t
# e
# d
 # M
# T
 # m
# o
# d
# e
# l
# s
 # a
# r
# e
 # s
# t
# i
# l
# l
 # n
# e
# c
# e
# s
# s
# a
# r
# y
# .
# "
# )


# e
# x
# c
# e
# p
# t
 # E
# n
# v
# i
# r
# o
# n
# m
# e
# n
# t
# E
# r
# r
# o
# r
 # a
# s
 # e
# :

    # p
# r
# i
# n
# t
# (
# f
# "
# K
# r
# u
# t
# r
# i
# m
 # k
# e
# y
 # n
# o
# t
 # s
# e
# t
# :
 # {
# e
# }
# "
# )

# e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
 # a
# s
 # e
# :

    # i
# m
# p
# o
# r
# t
 # t
# r
# a
# c
# e
# b
# a
# c
# k
# ;
 # t
# r
# a
# c
# e
# b
# a
# c
# k
# .
# p
# r
# i
# n
# t
# _
# e
# x
# c
# (
# )

